# Model training and validation

In [1]:
from pathlib import Path
import sys


In [2]:

import joblib
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.dummy import DummyClassifier


In [3]:

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
RAW_PATH = PROJECT_ROOT / "data/raw/data.csv"
assert RAW_PATH.is_file(), f"Missing raw data: {RAW_PATH}"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [4]:

from src.data import get_or_create_split
from src.evaluation import evaluate_predictions
from src.models import train_model
from src.schema import FEATURE_COLUMNS, ROW_ID, TARGET

In [5]:
MODEL_TO_USE = "auto"
THRESHOLD = 0.5
model_names = ("prior_baseline", "random_forest", "logistic_regression", "xgboost")

In [6]:
if MODEL_TO_USE != "auto" and MODEL_TO_USE not in model_names:
    raise ValueError(f"MODEL_TO_USE must be 'auto' or one of {model_names}.")


In [7]:
df = pd.read_csv(RAW_PATH)
outer_split = get_or_create_split(df, path=PROJECT_ROOT / "data/processed/split.parquet")
development = outer_split["train"]
inner_split = get_or_create_split(
    development.drop(columns=ROW_ID),
    path=PROJECT_ROOT / "data/processed/validation_split.parquet",
)
training = inner_split["train"]
validation = inner_split["test"]

display(pd.DataFrame({
    "rows": [len(training), len(validation)],
    "class_1_fraction": [training[TARGET].mean(), validation[TARGET].mean()],
}, index=["training", "validation"]))


,rows,class_1_fraction
training,331971,0.224128
validation,82993,0.224127


`train_model()` fits feature creation, cleaning, imputation, and the classifier together.
Logistic regression uses standard scaling; tree models use unscaled features.
Every learned transformation is fitted on training rows only.
Use the fixed threshold of 0.5 for threshold-dependent metrics.


In [ ]:
fitted_candidates = {}
validation_records = []

for name in model_names:
    print(f"Training and validating: {name}", flush=True)
    if name == "prior_baseline":
        pipeline = train_model(training, estimator=DummyClassifier(strategy="prior"))
    else:
        pipeline = train_model(training, model=name)
    probabilities = pipeline.predict_proba(validation.drop(columns=TARGET))[:, 1]
    metrics = evaluate_predictions(validation[TARGET], probabilities, threshold=THRESHOLD)
    fitted_candidates[name] = pipeline
    validation_records.append({"model": name, **metrics})


Training and validating: prior_baseline


Training and validating: random_forest


Training and validating: logistic_regression


Training and validating: xgboost


,roc_auc,average_precision,accuracy,balanced_accuracy,precision,recall,f1
model,,,,,,,
xgboost,0.715482,0.411497,0.781090,0.538362,0.567049,0.098436,0.167751
random_forest,0.712031,0.407303,0.656019,0.652108,0.353475,0.645019,0.456684
logistic_regression,0.704530,0.391875,0.655019,0.646608,0.350379,0.631364,0.450661
prior_baseline,0.500000,0.224127,0.775873,0.500000,0.000000,0.000000,0.000000


In [ ]:

validation_metrics = pd.DataFrame(validation_records).set_index("model")
validation_metrics = validation_metrics.sort_values("average_precision", ascending=False)
display(validation_metrics)


Set `MODEL_TO_USE` to select an algorithm explicitly, or keep `"auto"` for validation selection.
Refit every candidate on all development rows and save them in `models/loan_models.joblib`.
`models/loan_model.joblib` contains the selected pipeline for existing prediction code.
Test results do not influence selection.


In [9]:
validation_best_model = validation_metrics["average_precision"].idxmax()
selected_model = validation_best_model if MODEL_TO_USE == "auto" else MODEL_TO_USE
fitted_models = {}

for name, candidate in fitted_candidates.items():
    print(f"Refitting on development data: {name}", flush=True)
    fitted_models[name] = clone(candidate).fit(
        development.drop(columns=TARGET), development[TARGET],
    )

models_path = PROJECT_ROOT / "models"
reports_path = PROJECT_ROOT / "reports"
models_path.mkdir(parents=True, exist_ok=True)
reports_path.mkdir(parents=True, exist_ok=True)

metadata = {
    "selected_model": selected_model,
    "validation_best_model": validation_best_model,
    "threshold": THRESHOLD,
    "validation_metrics": validation_records,
    "feature_columns": list(FEATURE_COLUMNS),
    "selection_metric": "average_precision",
}
joblib.dump({"pipelines": fitted_models, **metadata}, models_path / "loan_models.joblib")
joblib.dump({"pipeline": fitted_models[selected_model], **metadata}, models_path / "loan_model.joblib")
validation_metrics.to_csv(reports_path / "validation_metrics.csv")

print(f"Highest validation average precision: {validation_best_model}")
print(f"Selected model: {selected_model}")
print(models_path / "loan_models.joblib")
print(models_path / "loan_model.joblib")


Refitting on development data: prior_baseline


Refitting on development data: random_forest


Refitting on development data: logistic_regression


Refitting on development data: xgboost


Highest validation average precision: xgboost
Selected model: xgboost
C:\Users\USER\Desktop\Portfolio\supervised\ml-anomaly-detection\models\loan_models.joblib
C:\Users\USER\Desktop\Portfolio\supervised\ml-anomaly-detection\models\loan_model.joblib
